# AssureX — Data Cleaning & Preparation

In [9]:

import pandas as pd
import numpy as np

TRAIN_PATH = "train.csv"
VAL_PATH = "validation.csv"
TEST_PATH = "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Original shapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


Original shapes:
Train: (1050, 43)
Validation: (225, 43)
Test: (225, 43)


In [10]:
print("Train info:", train_df.info(),"\n")
print("Train head:", train_df.head(),"\n")


<class 'pandas.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 43 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   claim_id                         1050 non-null   str    
 1   product_id                       1050 non-null   str    
 2   product_category                 1050 non-null   str    
 3   brand                            1050 non-null   str    
 4   model                            1050 non-null   str    
 5   serial_number                    1050 non-null   str    
 6   receipt_serial_number            1050 non-null   str    
 7   serial_number_status             1050 non-null   str    
 8   purchase_date                    1050 non-null   str    
 9   fault_date                       1050 non-null   str    
 10  claim_date                       1050 non-null   str    
 11  product_age_days                 1050 non-null   int64  
 12  warranty_duration_months       

In [11]:
print("Train describe:", train_df.describe(),"\n")
print("Train duplicated:", train_df.duplicated().sum(),"\n")

Train describe:        product_age_days  warranty_duration_months  warranty_remaining_days  \
count       1050.000000               1050.000000              1050.000000   
mean         108.979048                 18.034286               435.564762   
std          181.124755                  4.894187               208.561303   
min            5.000000                 12.000000                 0.000000   
25%           35.000000                 12.000000               315.000000   
50%           45.000000                 18.000000               495.000000   
75%           73.000000                 24.000000               644.750000   
max          765.000000                 24.000000               715.000000   

       previous_repair_count  previous_replacement_count  \
count            1050.000000                      1050.0   
mean                0.166667                         0.0   
std                 0.372856                         0.0   
min                 0.000000             

In [12]:
print( train_df.isnull().sum())

claim_id                              0
product_id                            0
product_category                      0
brand                                 0
model                                 0
serial_number                         0
receipt_serial_number                 0
serial_number_status                  0
purchase_date                         0
fault_date                            0
claim_date                            0
product_age_days                      0
warranty_duration_months              0
warranty_remaining_days               0
warranty_status                       0
fault_category                        0
fault_description                     0
physical_damage                       0
liquid_damage                         0
unauthorized_repair                   0
repair_history                        0
previous_repair_count                 0
last_repair_date                    875
previous_replacement                  0
previous_replacement_count            0


In [13]:

def clean_data(df):
    df = df.copy()

    # -------------------------------------------------
    # Remove high-cardinality identifiers
    # -------------------------------------------------
    ID_COLUMNS = [
        "claim_id",
        "product_id",
        "serial_number",
        "receipt_serial_number"
    ]

    df.drop(
        columns=[c for c in ID_COLUMNS if c in df.columns],
        inplace=True
    )

    # -------------------------------------------------
    # Remove data leakage columns
    # These represent downstream decisions/results.
    # -------------------------------------------------
    LEAKAGE_COLUMNS = [
        "hard_fail_detected",
        "warning_detected",
        "manual_review_trigger",
        "replacement_eligible"
    ]

    df.drop(
        columns=[c for c in LEAKAGE_COLUMNS if c in df.columns],
        inplace=True
    )

    # -------------------------------------------------
    # Remove zero-information historical columns
    # -------------------------------------------------
    CONSTANT_COLUMNS = [
        "previous_replacement",
        "previous_replacement_count",
        "previous_replacement_date"
    ]

    df.drop(
        columns=[c for c in CONSTANT_COLUMNS if c in df.columns],
        inplace=True
    )

    # -------------------------------------------------
    # Convert date columns
    # -------------------------------------------------
    DATE_COLUMNS = [
        "purchase_date",
        "fault_date",
        "claim_date",
        "last_repair_date"
    ]

    for col in DATE_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

    # -------------------------------------------------
    # Reporting days
    # claim_date - fault_date
    # -------------------------------------------------
    if "claim_date" in df.columns and "fault_date" in df.columns:
        df["reporting_days"] = (
            df["claim_date"] - df["fault_date"]
        ).dt.days

    # -------------------------------------------------
    # Previous repair information
    # -------------------------------------------------
    if "last_repair_date" in df.columns:
        df["has_previous_repair"] = (
            df["last_repair_date"].notna().astype(int)
        )

        if "claim_date" in df.columns:
            df["days_since_last_repair"] = (
                df["claim_date"] - df["last_repair_date"]
            ).dt.days

    # -------------------------------------------------
    # Missing documents
    # -------------------------------------------------
    if "missing_documents" in df.columns:
        df["missing_documents"] = (
            df["missing_documents"]
            .fillna("None")
            .astype(str)
            .str.strip()
        )

        if "missing_documents_count" not in df.columns:
            df["missing_documents_count"] = (
                df["missing_documents"].apply(
                    lambda x: 0
                    if x.lower() in ["none", "nan", ""]
                    else len([
                        item for item in x.split(",")
                        if item.strip()
                    ])
                )
            )

    # -------------------------------------------------
    # Boolean columns -> 0 / 1
    # -------------------------------------------------
    BOOLEAN_COLUMNS = [
        "physical_damage",
        "liquid_damage",
        "unauthorized_repair",
        "authorized_service_center",
        "receipt_available",
        "receipt_valid",
        "purchase_information_consistent",
        "supporting_evidence_available",
        "evidence_consistency",
        "duplicate_claim",
        "contradiction_detected",
        "claim_reporting_within_period",
        "replacement_requested"
    ]

    TRUE_VALUES = {
        True, 1, "1", "true", "True", "TRUE",
        "yes", "Yes", "YES"
    }

    for col in BOOLEAN_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: 1 if x in TRUE_VALUES else 0
            )

    # -------------------------------------------------
    # Remove raw date columns after feature creation
    # -------------------------------------------------
    df.drop(
        columns=[
            c for c in DATE_COLUMNS
            if c in df.columns
        ],
        inplace=True
    )

    # -------------------------------------------------
    # Remove free-text fault description.
    # Structured fault_category remains available.
    # -------------------------------------------------
    if "fault_description" in df.columns:
        df.drop(columns=["fault_description"], inplace=True)

    # -------------------------------------------------
    # Replace invalid numeric values with NaN.
    # Imputation is intentionally NOT done here.
    # -------------------------------------------------
    numeric_columns = df.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_columns:
        df[col] = df[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

    # -------------------------------------------------
    # Normalize missing categorical values to NaN.
    # -------------------------------------------------
    categorical_columns = df.select_dtypes(
        include=["object"]
    ).columns

    for col in categorical_columns:
        df[col] = df[col].replace(
            ["", "nan", "NaN", "None"],
            np.nan
        )

    return df


In [ ]:

# Apply exactly the same cleaning/feature engineering
# to all three datasets.

train_clean = clean_data(train_df)
val_clean = clean_data(val_df)
test_clean = clean_data(test_df)

print("Cleaned shapes:")
print("Train:", train_clean.shape)
print("Validation:", val_clean.shape)
print("Test:", test_clean.shape)



C:\Users\MF\AppData\Local\Temp\ipykernel_12868\1960700608.py:176: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(
C:\Users\MF\AppData\Local\Temp\ipykernel_12868\1960700608.py:176: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strin

Cleaned shapes:
Train: (1050, 30)
Validation: (225, 30)
Test: (225, 30)


C:\Users\MF\AppData\Local\Temp\ipykernel_12868\1960700608.py:176: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


In [15]:

# Separate target column.
# The target is never used as an input feature.

if "claim_class" not in train_clean.columns:
    raise ValueError("claim_class is missing from train.csv")

y_train = train_clean["claim_class"]
X_train = train_clean.drop(columns=["claim_class"])

if "claim_class" in val_clean.columns:
    y_val = val_clean["claim_class"]
    X_val = val_clean.drop(columns=["claim_class"])
else:
    y_val = None
    X_val = val_clean

if "claim_class" in test_clean.columns:
    y_test = test_clean["claim_class"]
    X_test = test_clean.drop(columns=["claim_class"])
else:
    y_test = None
    X_test = test_clean


In [16]:

# Ensure all datasets use exactly the same feature set and order.

train_features = set(X_train.columns)
val_features = set(X_val.columns)
test_features = set(X_test.columns)

if train_features != val_features:
    missing_in_val = sorted(train_features - val_features)
    extra_in_val = sorted(val_features - train_features)
    raise ValueError(
        f"Train/Validation feature mismatch. "
        f"Missing in validation: {missing_in_val}; "
        f"Extra in validation: {extra_in_val}"
    )

if train_features != test_features:
    missing_in_test = sorted(train_features - test_features)
    extra_in_test = sorted(test_features - train_features)
    raise ValueError(
        f"Train/Test feature mismatch. "
        f"Missing in test: {missing_in_test}; "
        f"Extra in test: {extra_in_test}"
    )

feature_order = X_train.columns.tolist()

X_train = X_train.reindex(columns=feature_order)
X_val = X_val.reindex(columns=feature_order)
X_test = X_test.reindex(columns=feature_order)

print("Feature consistency check: PASSED")
print("Number of final features:", len(feature_order))


Feature consistency check: PASSED
Number of final features: 29


In [17]:

# Final quality checks.
# Missing values are intentionally allowed here.
# They will be imputed later using statistics learned
# ONLY from the training set.

print("Remaining missing values:")
print("Train:", int(X_train.isna().sum().sum()))
print("Validation:", int(X_val.isna().sum().sum()))
print("Test:", int(X_test.isna().sum().sum()))

print("\nTarget distribution:")
print(y_train.value_counts())

print("\nFinal feature names:")
for i, col in enumerate(feature_order, start=1):
    print(f"{i:02d}. {col}")



Remaining missing values:
Train: 1759
Validation: 372
Test: 362

Target distribution:
claim_class
Valid Claim      350
Invalid Claim    350
Manual Review    350
Name: count, dtype: int64

Final feature names:
01. product_category
02. brand
03. model
04. serial_number_status
05. product_age_days
06. warranty_duration_months
07. warranty_remaining_days
08. warranty_status
09. fault_category
10. physical_damage
11. liquid_damage
12. unauthorized_repair
13. repair_history
14. previous_repair_count
15. authorized_service_center
16. receipt_available
17. receipt_valid
18. purchase_information_consistent
19. missing_documents
20. missing_documents_count
21. supporting_evidence_available
22. evidence_consistency
23. duplicate_claim
24. contradiction_detected
25. claim_reporting_within_period
26. replacement_requested
27. reporting_days
28. has_previous_repair
29. days_since_last_repair
